# Inject an external atmosphere in FGCM : final-cycle test

- author : Sylvie Dagoret-Campagne
- creation date : 2026-11-11
- follows : `01_runfgcmFitCyle.ipynb` (replay of step 2a). Not executed yet.

## What this notebook does, and what it does not

Goal : use atmospheric parameters measured elsewhere (AuxTel / libRadtran) instead of the ones fitted by `fgcmFitCycle`.

**Correction of notebook 01 (section 10, item 3).** It said that editing a `FitParameters` catalog and setting
`freezeStdAtmosphere=True` should keep the injected values. Reading the code of `fgcm` / `fgcmcal` shows that this is
**not true in a normal fit cycle** :

- in every cycle that is not the final one, `FgcmFitCycleTask` sets `resetFitParameters = True` (hard coded) ;
- `fgcm.FgcmFitCycle` then calls `resetAtmosphereParameters()` *before* the fit : `parAlpha`, `parO3`, `parLnTau*`,
  `parLnPwv*` of all nights are overwritten by the **standard** values (single numbers from the look-up table) ;
- the bounds of `freezeStdAtmosphere` are computed afterwards (`getParBounds`, in `_doFit`) : they freeze the parameters at
  these standard values. The name of the option is exact, it freezes the *standard* atmosphere.

What *is* possible without touching the code is the **final cycle** (`isFinalCycle=True`) : `maxIter = 0`,
`resetFitParameters = False`, the parameters read from the previous cycle are used as they are, and the zero points
are computed with them. So :

| mode | what it does |
|---|---|
| `null` | final cycle with the unmodified `Cycle K-1` parameters : must reproduce the final cycle of `stage2_v2` (validates the single-cycle path) |
| `closure` | final cycle with parameters rebuilt from "measurements" simulated from the fitted atmosphere (validates the conversion code below) |
| `aux` | final cycle with parameters built from the AuxTel table `AUX_FILE` |

This applies an injected atmosphere to the instrument parameters fitted *with the fitted atmosphere* : it is a
diagnostic (zero points and gray terms with the injected atmosphere), **not** the fit of the instrument alone. That
needs the reset to be replaced (see the last section).

## Model of the nightly parameters (`fgcm.FgcmParameters.parsToExposures`)

For an exposure taken on night `n` at `dUT` (days since the night boundary, `dUT = MJD + utBoundary - night`) :

- `ln PWV = parLnPwvIntercept[n] + parLnPwvSlope[n] dUT + parLnPwvQuadratic[n] dUT^2`  (PWV in mm)
- `ln tau = parLnTauIntercept[n] + parLnTauSlope[n] dUT`  (aerosol optical depth at 7750 A)
- `O3 = parO3[n]` (Dobson units), `alpha = parAlpha[n]`

with `night = floor(MJD + utBoundary)`. A per-visit measurement is therefore represented by a quadratic (PWV) or a
line (tau) in time over each night.

## 1. Configuration

In [ ]:
import copy
import os
import re
import shlex
import shutil
import subprocess
import logging
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from lsst.daf.butler import Butler
from lsst.pipe.base import Pipeline

logging.basicConfig(level=logging.WARNING, format='%(name)s %(levelname)s: %(message)s', force=True)
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_rows', 200)
print('python :', platform.python_version(), '| DRP_PIPE_DIR :', os.getenv('DRP_PIPE_DIR'))

In [ ]:
username = os.getenv('USER')
instrument = 'LSSTCam'

# ---- collections -------------------------------------------------------------------------------
RUN_NAME = 'runcosmos01_20261103_tract9813patchsel_RunCosmos9patches'
STAGE1 = f'u/{username}/LSSTCam/{RUN_NAME}/stage1'
STAGE2_REF = f'u/{username}/LSSTCam/{RUN_NAME}/stage2_v2'     # multi-cycle run of bps : all cycles are there
DEFAULTS = 'LSSTCam/defaults'

# ---- what to inject ----------------------------------------------------------------------------
INJECT_MODE = 'null'          # 'null' | 'closure' | 'aux'
AUX_FILE = None               # csv with columns mjd, pwv, tau, alpha, o3 (see section 4), needed for 'aux'

# ---- switches ----------------------------------------------------------------------------------
WRITE_INPUT = False           # True : write the edited FitParameters catalog in INJECT_RUN
EXECUTE = False               # True : run pipetask qgraph + run
INSPECT = 'reference'         # 'reference' : STAGE2_REF, 'new' : OUT_COLLECTION

INJECT_RUN = f'u/{username}/LSSTCam/{RUN_NAME}/fgcm_inject_input_{INJECT_MODE}'
OUT_COLLECTION = f'u/{username}/LSSTCam/{RUN_NAME}/fgcm_nb02_final_{INJECT_MODE}'
INSPECT_COLLECTION = STAGE2_REF if INSPECT == 'reference' else OUT_COLLECTION

PIPELINE_FILE = os.path.expandvars('${DRP_PIPE_DIR}/pipelines/LSSTCam/DRP.yaml')
NB_TAG = '02_injectAtmosphere'
DATA_DIR = Path(f'data_{NB_TAG}')
FIG_DIR = Path(f'figs_{NB_TAG}')
DATA_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

print('mode          :', INJECT_MODE)
print('inject run    :', INJECT_RUN)
print('output        :', OUT_COLLECTION)
print('inspecting    :', INSPECT_COLLECTION)

In [ ]:
butler = Butler('main')


def save_fig(fig, name):
    '''Save a figure as PNG and PDF in FIG_DIR.'''
    for ext in ('png', 'pdf'):
        fig.savefig(FIG_DIR / f'{name}.{ext}', bbox_inches='tight')


def query_refs(dataset_type, collections, find_first=True):
    '''Dataset references of a type (falls back to findFirst=False).'''
    if isinstance(collections, str):
        collections = [collections]
    try:
        return list(butler.registry.queryDatasets(dataset_type, collections=collections, findFirst=find_first))
    except Exception as exc:
        if find_first:
            return query_refs(dataset_type, collections, find_first=False)
        print(f'   {dataset_type}: query failed ({type(exc).__name__}: {str(exc)[:150]})')
        return []

## 2. Cycles of the bps run

`K` is the final cycle number of the multi-cycle run. The final cycle reads `fgcm_Cycle{K-1}_FitParameters` and
`fgcm_Cycle{K-1}_FlaggedStars` : these are the datasets that the single-cycle final run below will read, the first one
from `INJECT_RUN` when it is edited.

In [ ]:
pipeline = Pipeline.from_uri(PIPELINE_FILE + '#step2a-recalibrate-global')
pipeline.addConfigOverride('fgcmFitCycle', 'minStarPerExp', 100)      # as in stage2_v2
graph = pipeline.to_graph()
fit_config = graph.tasks['fgcmFitCycle'].config
out_config = graph.tasks['fgcmOutputProducts'].config

K = int(fit_config.multipleCyclesFinalCycleNumber)
ut_boundary = fit_config.utBoundary if fit_config.utBoundary is not None else 0.0
print('doMultipleCycles                :', fit_config.doMultipleCycles)
print('multipleCyclesFinalCycleNumber  :', K)
print('utBoundary                      :', ut_boundary)
print('freezeStdAtmosphere             :', fit_config.freezeStdAtmosphere)
print('fgcmOutputProducts cycleNumber  :', out_config.connections.cycleNumber)
assert str(K) == str(out_config.connections.cycleNumber), 'final cycle of fgcmOutputProducts differs from K'

rows = []
for cycle in range(K + 1):
    for name in ('FitParameters', 'FlaggedStars', 'ZeropointsCatalog', 'Zeropoints'):
        refs = query_refs(f'fgcm_Cycle{cycle}_{name}', STAGE2_REF, find_first=False)
        if refs:
            rows.append(dict(cycle=cycle, datasetType=f'fgcm_Cycle{cycle}_{name}', n=len(refs), run=refs[0].run))
cycles_table = pd.DataFrame(rows)
display(cycles_table)

## 3. Fitted atmosphere, night by night

The nights of fgcm are `np.unique(floor(MJD + utBoundary))` over the exposures of `fgcmVisitCatalog` : the same
construction is used here to attach the arrays of the `FitParameters` catalog to dates.

In [ ]:
visit_ref = query_refs('fgcmVisitCatalog', STAGE2_REF)[0]
vcat = butler.get(visit_ref)
visits = pd.DataFrame({'visit': [rec['visit'] for rec in vcat], 'mjd': [rec['mjd'] for rec in vcat]})
visits['night'] = np.floor(visits['mjd'] + ut_boundary).astype(int)
visits['dut'] = visits['mjd'] + ut_boundary - visits['night']
nights = np.sort(visits['night'].unique())
print(len(visits), 'visits,', len(nights), 'nights, MJD', nights.min(), '-', nights.max())

par_ref = query_refs(f'fgcm_Cycle{K - 1}_FitParameters', STAGE2_REF)[0]
par_cat = butler.get(par_ref)
PAR_NAMES = ['parAlpha', 'parO3', 'parLnTauIntercept', 'parLnTauSlope',
             'parLnPwvIntercept', 'parLnPwvSlope', 'parLnPwvQuadratic']
fitted = {name: np.asarray(par_cat[0][name], dtype=float).copy() for name in PAR_NAMES}
for name in PAR_NAMES:
    print(f'   {name:20s} size {fitted[name].size}')
assert fitted['parAlpha'].size == nights.size, 'number of nights differs from the size of the parameter arrays'

In [ ]:
def evaluate(params, night_index, dut):
    '''Atmosphere at exposures : night_index = index of the night, dut = days since the night boundary.'''
    ln_pwv = (params['parLnPwvIntercept'][night_index] + params['parLnPwvSlope'][night_index] * dut
              + params['parLnPwvQuadratic'][night_index] * dut ** 2)
    ln_tau = params['parLnTauIntercept'][night_index] + params['parLnTauSlope'][night_index] * dut
    return pd.DataFrame(dict(pwv=np.exp(ln_pwv), tau=np.exp(ln_tau),
                             alpha=params['parAlpha'][night_index], o3=params['parO3'][night_index]))


visits['night_index'] = np.searchsorted(nights, visits['night'])
atm_fitted = evaluate(fitted, visits['night_index'].values, visits['dut'].values)
atm_fitted.insert(0, 'mjd', visits['mjd'].values)
display(atm_fitted.describe().T)
atm_fitted.to_csv(DATA_DIR / 'atmosphere_fitted_at_visits.csv', index=False)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
for ax, col, label in zip(axes.ravel(), ['pwv', 'tau', 'alpha', 'o3'],
                          ['PWV (mm)', 'aerosol tau (7750 A)', 'alpha', 'O3 (DU)']):
    ax.plot(atm_fitted['mjd'], atm_fitted[col], '.', ms=3)
    ax.set_ylabel(label)
    ax.set_title(f'fitted {col}, cycle {K - 1} of stage2_v2')
for ax in axes[1]:
    ax.set_xlabel('MJD')
fig.tight_layout()
save_fig(fig, 'atmosphere_fitted_vs_mjd')
plt.show()

## 4. Convert measurements into nightly parameters

`AUX_FILE` : csv with one row per measurement and the columns `mjd`, `pwv` (mm), `tau` (aerosol optical depth **at 7750 A**,
to be converted from the wavelength of the AuxTel fit), `alpha`, `o3` (DU). Any column can be missing or NaN. For each
night of fgcm :

- PWV : polynomial of degree 2 in `dUT` on `ln PWV` (degree 1 or 0 if there are fewer than 3 or 2 measurements) ;
- tau : polynomial of degree 1 on `ln tau` ;
- alpha, O3 : median over the night ;
- a night without measurement keeps the fitted values.

The `closure` mode builds `aux` from the fitted atmosphere at each visit : the evaluated atmosphere at the visits must be
recovered, which checks the conversion.

In [ ]:
def poly_coeffs(dut, y, max_degree):
    '''Coefficients in increasing order (a, b, c) of a polynomial fit of degree <= max_degree.'''
    good = np.isfinite(dut) & np.isfinite(y)
    dut, y = np.asarray(dut)[good], np.asarray(y)[good]
    degree = min(max_degree, dut.size - 1, np.unique(dut).size - 1)
    if dut.size == 0:
        return None
    if degree < 0:
        return None
    coeffs = np.polynomial.polynomial.polyfit(dut, y, degree) if degree > 0 else np.array([np.mean(y)])
    out = np.zeros(max_degree + 1)
    out[:coeffs.size] = coeffs
    return out


def nightly_from_aux(aux, nights, fitted, ut_boundary):
    '''Nightly parameters of fgcm from measurements. Returns (parameters, number of measurements per night).'''
    new = {name: arr.copy() for name, arr in fitted.items()}
    count = np.zeros(nights.size, dtype=int)
    night_of = np.floor(aux['mjd'].values + ut_boundary).astype(int)
    dut = aux['mjd'].values + ut_boundary - night_of
    for i, night in enumerate(nights):
        sel = night_of == night
        count[i] = sel.sum()
        if not sel.any():
            continue
        if 'pwv' in aux:
            y = np.log(np.where(aux['pwv'].values[sel] > 0, aux['pwv'].values[sel], np.nan))
            c = poly_coeffs(dut[sel], y, 2)
            if c is not None:
                new['parLnPwvIntercept'][i], new['parLnPwvSlope'][i], new['parLnPwvQuadratic'][i] = c
        if 'tau' in aux:
            y = np.log(np.where(aux['tau'].values[sel] > 0, aux['tau'].values[sel], np.nan))
            c = poly_coeffs(dut[sel], y, 1)
            if c is not None:
                new['parLnTauIntercept'][i], new['parLnTauSlope'][i] = c
        for col, par in (('alpha', 'parAlpha'), ('o3', 'parO3')):
            if col in aux and np.isfinite(aux[col].values[sel]).any():
                new[par][i] = np.nanmedian(aux[col].values[sel])
    return new, count


def closure_test(atm, visits, nights, fitted, ut_boundary):
    '''Rebuild the parameters from the atmosphere at the visits, and compare the atmosphere at the visits.'''
    new, count = nightly_from_aux(atm, nights, fitted, ut_boundary)
    back = evaluate(new, visits['night_index'].values, visits['dut'].values)
    rel = {col: np.nanmax(np.abs(back[col].values / atm[col].values - 1)) for col in ('pwv', 'tau', 'alpha', 'o3')}
    return new, count, rel

In [ ]:
if INJECT_MODE == 'null':
    injected, n_aux = {name: arr.copy() for name, arr in fitted.items()}, np.zeros(nights.size, dtype=int)
elif INJECT_MODE == 'closure':
    injected, n_aux, rel = closure_test(atm_fitted, visits, nights, fitted, ut_boundary)
    print('max relative difference at the visits :', {k: f'{v:.2e}' for k, v in rel.items()})
elif INJECT_MODE == 'aux':
    aux = pd.read_csv(AUX_FILE)
    injected, n_aux = nightly_from_aux(aux, nights, fitted, ut_boundary)
    print(f'{(n_aux > 0).sum()} nights out of {nights.size} have measurements')
else:
    raise ValueError(INJECT_MODE)

atm_injected = evaluate(injected, visits['night_index'].values, visits['dut'].values)
atm_injected.insert(0, 'mjd', visits['mjd'].values)

fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
for ax, col, label in zip(axes.ravel(), ['pwv', 'tau', 'alpha', 'o3'],
                          ['PWV (mm)', 'aerosol tau (7750 A)', 'alpha', 'O3 (DU)']):
    ax.plot(atm_fitted['mjd'], atm_fitted[col], '.', ms=3, label='fitted (stage2_v2)')
    ax.plot(atm_injected['mjd'], atm_injected[col], '.', ms=2, label=f'injected ({INJECT_MODE})')
    ax.set_ylabel(label)
axes[0, 0].legend()
for ax in axes[1]:
    ax.set_xlabel('MJD')
fig.tight_layout()
save_fig(fig, f'atmosphere_injected_{INJECT_MODE}')
plt.show()

## 5. Write the edited `FitParameters`

The edited catalog is written with the name `fgcm_Cycle{K-1}_FitParameters` in a new run (`INJECT_RUN`) that is chained
*before* `stage2_v2` in the input collections : the final-cycle task finds the edited catalog first. Nothing is written
unless `WRITE_INPUT = True`.

In [ ]:
def make_injected_catalog(cat, new):
    '''Deep copy of the FitParameters catalog with the nightly atmosphere arrays replaced.'''
    out = cat.copy(deep=True)
    rec = out[0]
    for name, arr in new.items():
        rec[name][:] = arr
        if not np.allclose(np.asarray(rec[name], dtype=float), arr):
            raise RuntimeError(f'could not set the field {name}')
    return out


injected_cat = make_injected_catalog(par_cat, injected)
dataset_name = f'fgcm_Cycle{K - 1}_FitParameters'

if WRITE_INPUT:
    if query_refs(dataset_name, INJECT_RUN, find_first=False):
        print(f'{dataset_name} already exists in {INJECT_RUN} : nothing written')
    else:
        wbutler = Butler('main', writeable=True, run=INJECT_RUN)
        wbutler.put(injected_cat, dataset_name, instrument=instrument)
        print(f'{dataset_name} written in {INJECT_RUN}')
else:
    print(f'WRITE_INPUT = False : {dataset_name} not written')

## 6. Final cycle : `fgcmFitCycle` + `fgcmOutputProducts`

Only the two last tasks are run (`DRP.yaml#fgcmFitCycle,fgcmOutputProducts`) : `fgcmBuildFromIsolatedStar` is not
rerun, its outputs (`fgcmVisitCatalog`, star catalogs) are read from `stage2_v2`. `fgcmFitCycle` is configured as a
single final cycle in a config file written below.

In [ ]:
config_file = DATA_DIR / 'fgcmFitCycle_finalCycle.py'
config_file.write_text(
    "# Single final cycle : reads fgcm_Cycle{K-1}_FitParameters / FlaggedStars, no fit (maxIter = 0)\n"
    "config.doMultipleCycles = False\n"
    f"config.cycleNumber = {K}\n"
    "config.isFinalCycle = True\n"
    f"config.connections.cycleNumber = '{K}'\n"
    f"config.connections.previousCycleNumber = '{K - 1}'\n"
    "config.minStarPerExp = 100\n")

input_collections = [INJECT_RUN, STAGE2_REF, STAGE1, DEFAULTS]
pipeline_uri = PIPELINE_FILE + '#fgcmFitCycle,fgcmOutputProducts'
qgraph_file = DATA_DIR / 'final_cycle.qgraph'

cmd_qgraph = ['pipetask', 'qgraph', '-b', 'main', '-i', ','.join(input_collections), '-o', OUT_COLLECTION,
              '-p', pipeline_uri, '-C', f'fgcmFitCycle:{config_file}', '-d', f"instrument='{instrument}'",
              '--save-qgraph', str(qgraph_file)]
cmd_run = ['pipetask', 'run', '-b', 'main', '-g', str(qgraph_file), '--register-dataset-types', '-j', '1']
print(shlex.join(cmd_qgraph))
print()
print(shlex.join(cmd_run))

In [ ]:
def run_cmd(cmd, log_path):
    '''Run a command, write the full log in log_path and print only the lines that matter.'''
    if shutil.which(cmd[0]) is None:
        raise RuntimeError(f'{cmd[0]} not found in PATH')
    print('$', shlex.join(cmd))
    keep = ('ERROR', 'FATAL', 'Traceback', 'Error', 'quanta', 'Quantum', 'graph')
    with open(log_path, 'w') as log:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in proc.stdout:
            log.write(line)
            if any(word in line for word in keep):
                print(line.rstrip()[:300])
        returncode = proc.wait()
    print(f'return code : {returncode} ; full log : {log_path}')
    if returncode != 0:
        raise RuntimeError(f'{cmd[1]} failed, see {log_path}')


if EXECUTE:
    if not query_refs(dataset_name, INJECT_RUN, find_first=False):
        raise RuntimeError(f'{dataset_name} is not in {INJECT_RUN} : set WRITE_INPUT = True first')
    run_cmd(cmd_qgraph, DATA_DIR / f'pipetask_qgraph_{INJECT_MODE}.log')
    run_cmd(cmd_run, DATA_DIR / f'pipetask_run_{INJECT_MODE}.log')
    butler = Butler('main')
else:
    print('EXECUTE = False : nothing run')

## 7. Effect on the zero points

`fgcmPhotoCalibCatalog` of the new run against the one of `stage2_v2`. In `null` mode the two must be identical : this
validates the single-cycle final path. In the other modes the difference is the effect of the injected atmosphere.

In [ ]:
def zeropoints(collection):
    '''Per (visit, detector) zero point of fgcmPhotoCalibCatalog : zp = 31.4 - 2.5 log10(calibrationMean).'''
    rows = []
    for ref in query_refs('fgcmPhotoCalibCatalog', collection):
        for rec in butler.get(ref):
            photo_calib = rec.getPhotoCalib()
            if photo_calib is not None:
                rows.append(dict(visit=ref.dataId['visit'], detector=rec.getId(),
                                 calib_mean=photo_calib.getCalibrationMean()))
    df = pd.DataFrame(rows)
    if len(df):
        df['zp'] = np.where(df['calib_mean'] > 0, 31.4 - 2.5 * np.log10(df['calib_mean']), np.nan)
    return df


if EXECUTE or INSPECT == 'new':
    zp_ref = zeropoints(STAGE2_REF)
    zp_new = zeropoints(OUT_COLLECTION)
    both = zp_ref.merge(zp_new, on=['visit', 'detector'], suffixes=('_ref', '_new'))
    both['dzp_mmag'] = 1000 * (both['zp_new'] - both['zp_ref'])
    both = both.merge(visits[['visit', 'mjd']], on='visit', how='left')
    print(len(zp_ref), 'reference,', len(zp_new), 'new,', len(both), 'in common')
    print('new - reference (mmag) : median', both['dzp_mmag'].median(), ', std', both['dzp_mmag'].std(),
          ', max |.|', both['dzp_mmag'].abs().max())

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    axes[0].hist(both['dzp_mmag'].dropna(), bins=100)
    axes[0].set_xlabel('new - reference (mmag)')
    axes[1].plot(both['mjd'], both['dzp_mmag'], '.', ms=2)
    axes[1].set_xlabel('MJD')
    axes[1].set_ylabel('new - reference (mmag)')
    fig.suptitle(f'final cycle with injected atmosphere : {INJECT_MODE}')
    fig.tight_layout()
    save_fig(fig, f'zeropoints_{INJECT_MODE}_vs_reference')
    plt.show()
else:
    print('Nothing to compare yet : run section 6 with EXECUTE = True, then INSPECT = "new"')

## 8. Fitting the instrument with a frozen external atmosphere : what is missing

The final-cycle test above does not refit the instrument. To fit only the instrumental parameters with the atmosphere
fixed at the injected values, two things must change inside a *non-final* cycle :

1. `resetAtmosphereParameters()` must not overwrite the nightly arrays with the standard values (`resetFitParameters`
   is hard coded to `True` in `FgcmFitCycleTask` for non-final cycles, and there is no configuration for it) ;
2. the bounds of `freezeStdAtmosphere` are then computed from the injected values (they already use the *current* values
   of the parameters, so item 1 is enough).

A minimal way, without copying `fgcmcal` : a small Python module defining a subclass of `FgcmFitCycleTask` that replaces
`FgcmParameters.resetAtmosphereParameters` by a function writing the injected nightly arrays (read from a file), used
through a copy of the pipeline yaml with `fgcmFitCycle: class: <module>.<Task>`. It is not written yet : to be done
once the final-cycle test above is understood, and it requires that the module is importable by the process running the
task (`PYTHONPATH`).

Other open questions : the conversion of the AuxTel aerosol fit to tau at 7750 A, the time sampling of the AuxTel
measurements compared to the quadratic in `dUT`, and the nights without AuxTel data (they keep the fitted values here).